In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml
python3: can't open file '/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/scripts/clean_skills.py': [Errno 2] No such file or directory


In [ ]:
from pathlib import Path
import re
import pandas as pd

# ---------- paths ----------
try:
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    BASE_DIR = Path.cwd()

PROCESSED_DIR = BASE_DIR / "data" / "processed"
IN_PATH = PROCESSED_DIR / "skills.csv"
OUT_PATH = PROCESSED_DIR / "skills_cleaned.csv"
FLAGGED_PATH = PROCESSED_DIR / "skills_flagged_for_review.csv"


# ---------- helper functions ----------

TITLE_WORDS = {
    "developer", "engineer", "architect", "manager", "consultant",
    "analyst", "lead", "leader", "specialist", "administrator",
    "admin", "tester", "designer", "qa"
}

MANUAL_CANONICAL_MAP = {
    "js": "javascript",
    "java script": "javascript",
    "react js": "react",
    "reactjs": "react",
    "node js": "node.js",
    "nodejs": "node.js",
    "c sharp": "c#",
    "c# .net": "c#",
    ".net": ".net",
    "ms sql": "sql",
    "mysql": "sql",
    "postgresql": "sql",
    "postgre sql": "sql",
    "mssql": "sql",
}


def strip_title_words(name: str) -> str:
    """
    Remove job-title words like 'developer', 'engineer' if there are other words.
    e.g. 'java developer' -> 'java'
    """
    words = name.split()
    if len(words) <= 1:
        return name
    cleaned = [w for w in words if w not in TITLE_WORDS]
    if cleaned:
        return " ".join(cleaned)
    # if everything was stripped, keep original
    return name


def canonicalize_name(raw_name: str) -> str:
    """Lowercase, trim, apply manual mappings and remove title words."""
    if not isinstance(raw_name, str):
        return ""

    name = raw_name.strip().lower()
    name = re.sub(r"\s+", " ", name)

    # apply manual canonical overrides
    if name in MANUAL_CANONICAL_MAP:
        name = MANUAL_CANONICAL_MAP[name]

    # remove job-title words (developer, engineer, etc.)
    name = strip_title_words(name)

    return name


def guess_category(name: str) -> str:
    n = name.lower()

    frontend = ["html", "css", "javascript", "js", "react", "angular", "vue", "frontend"]
    backend = ["java", "spring", "c#", ".net", "node", "django", "flask", "laravel", "php", "backend", "ruby"]
    database = ["sql", "mysql", "postgres", "oracle", "mongodb", "database", "nosql"]
    devops = ["docker", "kubernetes", "jenkins", "ci", "cd", "ansible", "terraform", "devops"]
    cloud = ["aws", "azure", "gcp", "google cloud", "cloud"]
    mobile = ["android", "kotlin", "ios", "swift", "react native", "flutter"]
    ml_ai = ["machine learning", "deep learning", "pytorch", "tensorflow", "ml", "ai", "nlp"]
    analytics = ["excel", "power bi", "tableau", "analytics", "data analysis", "reporting"]

    soft = ["communication", "teamwork", "team player", "leadership", "problem solving", "negotiation"]

    for kw in soft:
        if kw in n:
            return "soft_skill"
    for kw in frontend:
        if kw in n:
            return "frontend"
    for kw in backend:
        if kw in n:
            return "backend"
    for kw in database:
        if kw in n:
            return "database"
    for kw in devops:
        if kw in n:
            return "devops"
    for kw in cloud:
        if kw in n:
            return "cloud"
    for kw in mobile:
        if kw in n:
            return "mobile"
    for kw in ml_ai:
        if kw in n:
            return "ml_ai"
    for kw in analytics:
        if kw in n:
            return "analytics"

    return "other"


def guess_type(category: str, name: str) -> str:
    if category == "soft_skill":
        return "soft"
    if any(k in name for k in ["communication", "teamwork", "problem solving"]):
        return "soft"
    return "technical"


def is_job_title_like(raw_name: str) -> bool:
    """
    Heuristic: multi-word phrase ending with a title word, e.g. 'java developer'.
    """
    name = raw_name.strip().lower()
    words = name.split()
    if len(words) <= 1:
        return False
    # if any of the words is a title word, flag it
    return any(w in TITLE_WORDS for w in words)


# ---------- main cleaning routine ----------

def main():
    if not IN_PATH.exists():
        raise FileNotFoundError(f"{IN_PATH} not found")

    df = pd.read_csv(IN_PATH)

    # normalize names and compute canonical form
    df["name_raw"] = df["name"].astype(str)
    df["name_norm"] = df["name_raw"].str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
    df["name_canonical"] = df["name_norm"].apply(canonicalize_name)

    # re-guess category & type based on canonical name
    df["category_clean"] = df["name_canonical"].apply(guess_category)
    df["type_clean"] = df.apply(lambda r: guess_type(r["category_clean"], r["name_canonical"]), axis=1)

    # mark likely job titles / suspicious entries
    df["is_job_title_like"] = df["name_raw"].apply(is_job_title_like)
    df["is_other_category"] = df["category_clean"] == "other"
    df["flag_for_review"] = df["is_job_title_like"] | df["is_other_category"]

    # group by canonical name to merge duplicates
    groups = df.groupby("name_canonical", dropna=True)

    cleaned_rows = []
    flagged_rows = []

    for canon, g in groups:
        if not canon:   # skip empty
            continue

        # choose first skill_id as representative
        first = g.iloc[0]
        skill_id = first["skill_id"]

        # collect all name variants as aliases
        aliases = sorted(set(g["name_raw"].tolist()))
        aliases_str = ";".join(aliases)

        category = first["category_clean"]
        stype = first["type_clean"]

        row = {
            "skill_id": skill_id,
            "name": canon,
            "aliases": aliases_str,
            "category": category,
            "type": stype,
        }
        cleaned_rows.append(row)

        # if any of the group rows were flagged, add to flagged list for human review
        if g["flag_for_review"].any():
            fr = row.copy()
            fr["examples"] = aliases_str
            flagged_rows.append(fr)

    cleaned_df = pd.DataFrame(cleaned_rows).sort_values("name").reset_index(drop=True)
    flagged_df = pd.DataFrame(flagged_rows).sort_values("name").reset_index(drop=True)

    cleaned_df.to_csv(OUT_PATH, index=False)
    flagged_df.to_csv(FLAGGED_PATH, index=False)

    print(f"Saved cleaned skills to   {OUT_PATH}  ({len(cleaned_df)} rows)")
    print(f"Flagged {len(flagged_df)} skills for manual review at {FLAGGED_PATH}")


if __name__ == "__main__":
    main()


Saved cleaned skills to   /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/processed/skills_cleaned.csv  (285 rows)
Flagged 225 skills for manual review at /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/processed/skills_flagged_for_review.csv
